# FYP: Robust Deep Image Steganography Against Downscaling
Colab starter notebook. Run cells top to bottom.

**First: Runtime -> Change runtime type -> T4 GPU -> Save**

In [2]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - go enable the GPU runtime!')

CUDA available: True
Device: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Clone your repo
Push the `fyp_stego/` folder (this notebook's sibling files) to a GitHub repo first, then clone it here.
If you haven't made the repo yet, skip this cell and use the next one to upload files manually instead.

In [4]:
!git clone https://github.com/syakir12343425/FYP-Project.git
%cd FYP-Project

Cloning into 'FYP-Project'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 25 (delta 2), reused 20 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 47.20 KiB | 409.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/FYP-Project


In [5]:
!python models.py


Encoder params: 105,321
Decoder params: 104,161
stego: torch.Size([2, 3, 512, 512]) 0.0 1.0
recovered: torch.Size([2, 1024])
BER on untrained model (sanity check, expect ~0.5): 0.496


In [6]:
!pip install -q pytorch-msssim scikit-image opencv-python-headless pyyaml tqdm pandas

## Download DIV2K straight into Drive (do this once; it persists across sessions)

## Point config at Drive paths, set checkpoint dir to Drive too (survives disconnects)

In [8]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['train_dir'] = f'{LOCAL_DATA}/DIV2K_train_HR'
cfg['data']['val_dir'] = f'{LOCAL_DATA}/DIV2K_valid_HR'

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(cfg['data'])

NameError: name 'LOCAL_DATA' is not defined

## Quick sanity check (shapes, no training)

## Train
Runs for up to `train.epochs` (default 60), checkpointing every epoch to Drive.
If the session disconnects, re-run this cell with `--resume` pointing at `last.pt`.

In [ ]:
!python train.py --config configs/config.yaml --resume

In [ ]:
import os
os.makedirs(f'{LOCAL_DATA}', exist_ok=True)
!unzip -q {DATA_DIR}/DIV2K_valid_HR.zip -d {LOCAL_DATA}

n = len(os.listdir(f'{LOCAL_DATA}/DIV2K_valid_HR'))
print(f"valid images: {n}")
assert n == 100, f"expected 100, got {n} — the zip itself may be incomplete, redownload it"



In [ ]:
# To resume after a disconnect, uncomment:
# !python train.py --config configs/config.yaml --resume /content/drive/MyDrive/fyp_checkpoints/last.pt

## Full evaluation (5 algorithms x 4 sizes x 100 images, CNN vs LSB)

In [ ]:
!python eval.py --config configs/config.yaml --checkpoint /content/drive/MyDrive/fyp_checkpoints/best.pt --out-dir /content/drive/MyDrive/fyp_results

In [ ]:
import pandas as pd
summary = pd.read_csv('/content/drive/MyDrive/fyp_results/eval_summary.csv')
summary